In [1]:
import pandas as pd
import customtkinter as ctk
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# 1. إعداد النموذج
df = pd.read_csv('processed.csv')
df.columns = df.columns.str.strip()
le = LabelEncoder()
df['GENDER'] = le.fit_transform(df['GENDER'])
df['LUNG_CANCER'] = le.fit_transform(df['LUNG_CANCER'])
features = ['GENDER', 'AGE', 'SMOKING', 'YELLOW_FINGERS', 'ANXIETY', 'PEER_PRESSURE', 
            'CHRONIC DISEASE', 'FATIGUE', 'ALLERGY', 'WHEEZING', 'ALCOHOL CONSUMING', 
            'COUGHING', 'SHORTNESS OF BREATH', 'SWALLOWING DIFFICULTY', 'CHEST PAIN', 'SMOKING_PACK_YEARS']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(df[features], df['LUNG_CANCER'])

# 2. الواجهة الاحترافية
ctk.set_appearance_mode("Dark")
ctk.set_default_color_theme("blue")

class ProMedicalApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("PRO-ANALYTICS | Clinical AI Dashboard")
        self.geometry("1100x750")
        
        # اللوحة الجانبية للشعار والبيانات
        self.sidebar = ctk.CTkFrame(self, width=300, fg_color="#1a1a1a")
        self.sidebar.pack(side="left", fill="y")
        
        ctk.CTkLabel(self.sidebar, text="🩺 PRO-MED AI", font=("Roboto", 28, "bold")).pack(pady=40)
        
        self.scroll = ctk.CTkScrollableFrame(self.sidebar, fg_color="transparent")
        self.scroll.pack(fill="both", expand=True, padx=10)
        
        self.inputs = {}
        for f in features:
            frame = ctk.CTkFrame(self.scroll, fg_color="#2b2b2b", corner_radius=8)
            frame.pack(fill="x", pady=5, padx=5)
            ctk.CTkLabel(frame, text=f.replace("_", " ").title(), font=("Arial", 12)).pack(side="left", padx=10)
            if f == 'GENDER': self.inputs[f] = ctk.CTkOptionMenu(frame, values=["0", "1"], width=80)
            elif f in ['AGE', 'SMOKING_PACK_YEARS']: self.inputs[f] = ctk.CTkEntry(frame, width=80)
            else: self.inputs[f] = ctk.CTkSwitch(frame, text="", width=40)
            self.inputs[f].pack(side="right", padx=10)

        # اللوحة الرئيسية للنتائج
        self.main_view = ctk.CTkFrame(self, fg_color="transparent")
        self.main_view.pack(side="right", fill="both", expand=True, padx=30, pady=30)
        
        ctk.CTkLabel(self.main_view, text="DIAGNOSTIC ANALYSIS ENGINE", font=("Roboto", 24, "bold")).pack(anchor="w")
        
        self.res_card = ctk.CTkFrame(self.main_view, height=300, fg_color="#2b2b2b")
        self.res_card.pack(fill="x", pady=20)
        
        self.res_label = ctk.CTkLabel(self.res_card, text="Awaiting Input...", font=("Arial", 40, "bold"))
        self.res_label.pack(pady=50)
        
        self.prog_bar = ctk.CTkProgressBar(self.main_view, height=30)
        self.prog_bar.pack(fill="x", pady=20)
        self.prog_label = ctk.CTkLabel(self.main_view, text="Risk Percentage: 0%", font=("Arial", 20))
        self.prog_label.pack()
        
        ctk.CTkButton(self.main_view, text="EXECUTE DIAGNOSIS", height=60, fg_color="#3498db", 
                      font=("Arial", 18, "bold"), command=self.analyze).pack(pady=40, fill="x")

    def analyze(self):
        data = [float(self.inputs[f].get()) if not isinstance(self.inputs[f], ctk.CTkSwitch) 
                else (2.0 if self.inputs[f].get() else 1.0) for f in features]
        prob = model.predict_proba([data])[0][1]
        
        # تحديث النتيجة
        res = "POSITIVE" if prob > 0.5 else "NEGATIVE"
        color = "#e74c3c" if prob > 0.5 else "#27ae60"
        
        self.res_label.configure(text=res, text_color=color)
        self.prog_bar.set(prob)
        self.prog_bar.configure(progress_color=color)
        self.prog_label.configure(text=f"Risk Percentage: {prob*100:.1f}%")

if __name__ == "__main__":
    app = ProMedicalApp()
    app.mainloop()